# Этап 11 V1 — RealMLP против GBDT_mean

## Исследовательский вопрос

Может ли pre-tuned RealMLP на тех же 47 разрешённых признаках дать material improvement относительно принятого B*=GBDT_mean?

Проверяется ровно одна модель: официальный PyTabKit RealMLP_TD_Classifier. Это tuned-default recipe, а не HPO, не ensemble и не AutoGluon wrapper.

Неизменны dataset Data_final.xlsb, target DefMark, identifier INN, рабочая выборка из 289614 строк, порядок строк, 47 признаков, outer StratifiedKFold(3, shuffle=True, random_state=42), seeds 43/44/45 и final test. Q_B1_norm и Q_B2_norm не являются predictors. GBDT заново не обучается: baseline берётся из сохранённого leakage-safe Stage 7 OOF.

Параметры, фиксируемые для controlled outer-fold protocol: device=cpu, n_cv=1, n_refit=0, n_ens=1 и fold-specific random_state. Никаких HPO, bagging/ensembling, calibration, class weighting, balancing, sampling или threshold optimization нет. Native preprocessing и внутреннее validation/early stopping RealMLP обучаются только на соответствующем outer-train fold.

Primary selection metric — полный OOF Gini. Precision, Recall и F1 при 0.5 являются только диагностикой. Decision rule зафиксирован до запуска: material_gain, если ΔGini >= +0.010 и RealMLP выигрывает Gini минимум на 2/3 folds; inferior, если ΔGini <= -0.010 и проигрывает минимум на 2/3; иначе no_material_benefit.


### Что проверяем?

Подготавливаем воспроизводимый contract: пути, feature identity, hashes, CV и output artifacts. Это нужно сделать до чтения данных и до обучения, чтобы experiment не мог молча изменить locked design. Неизменными остаются все Stage 1–10 artifacts.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
from pytabkit import RealMLP_TD_Classifier
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent
GENERATED, SUMMARY = ROOT / 'reports' / 'generated', ROOT / 'reports' / 'summary'
DATASET = ROOT / 'data' / 'raw' / 'Data_final.xlsb'
STAGE1_PATH = GENERATED / 'stage1_baseline_results_V2.json'
STAGE7_PATH = GENERATED / 'stage7_tabm_stacking_results_V1.json'
STAGE7_OOF_PATH = GENERATED / 'stage7_tabm_stacking_oof_V1.npz'
RESULT_PATH = GENERATED / 'stage11_realmlp_results_V1.json'
OOF_PATH = GENERATED / 'stage11_realmlp_oof_V1.npz'
SUMMARY_PATH = SUMMARY / 'stage11_realmlp_summary_V1.json'

TARGET, IDENTIFIER = 'DefMark', 'INN'
FORBIDDEN = ('Q_B1_norm', 'Q_B2_norm')
EXPECTED_DATASET_SHA = 'fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930'
EXPECTED_WORKING_SHA = '80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45'
EXPECTED_N, OUTER_SEED, FOLD_SEEDS, THRESHOLD = 289614, 42, (43, 44, 45), 0.5
MODEL_OVERRIDES = {'device': 'cpu', 'n_cv': 1, 'n_refit': 0, 'n_ens': 1, 'verbosity': 2}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def sha256_indices(indices: np.ndarray) -> str:
    return hashlib.sha256(np.asarray(indices, dtype=np.int64).tobytes()).hexdigest()

def metrics_at_threshold(target: np.ndarray, probability: np.ndarray) -> dict[str, float]:
    predicted = (probability >= THRESHOLD).astype(np.int8)
    auc = float(roc_auc_score(target, probability))
    return {'ROC-AUC': auc, 'Gini': 2.0 * auc - 1.0, 'PR-AUC': float(average_precision_score(target, probability)), 'Precision': float(precision_score(target, predicted, zero_division=0)), 'Recall': float(recall_score(target, predicted, zero_division=0)), 'F1': float(f1_score(target, predicted, zero_division=0))}

def json_safe(value):
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return repr(value)

def atomic_json(path: Path, payload: dict) -> None:
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False, suffix='.tmp') as stream:
        json.dump(payload, stream, ensure_ascii=False, indent=2, allow_nan=False)
    os.replace(stream.name, path)

def atomic_npz(path: Path, **arrays: np.ndarray) -> None:
    with tempfile.NamedTemporaryFile('wb', dir=path.parent, delete=False, suffix='.tmp') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(stream.name, path)

print(f'PyTabKit: {importlib.metadata.version("pytabkit")}; device: cpu')
print(f'Output artifacts: {RESULT_PATH.name}, {OOF_PATH.name}, {SUMMARY_PATH.name}')


### Что проверяем?

Проверяем preflight до обучения: идентичность dataset и working indices, target/fold alignment, baseline GBDT_mean и набор ровно из 47 разрешённых features. Это непосредственно исключает leakage через validation fold и prevents training при несоответствии принятому control. Никакая preprocessing information из outer-validation здесь не создаётся.


In [ ]:
stage1 = json.loads(STAGE1_PATH.read_text(encoding='utf-8'))
stage7 = json.loads(STAGE7_PATH.read_text(encoding='utf-8'))
assert DATASET.exists() and STAGE7_OOF_PATH.exists(), 'STOP: отсутствует обязательный input artifact'
assert sha256_file(DATASET) == EXPECTED_DATASET_SHA, 'STOP: dataset SHA mismatch'
features = list(stage1['допустимые_признаки'])
assert len(features) == 47 and len(set(features)) == 47, 'STOP: feature count/uniqueness mismatch'
assert not set(FORBIDDEN).intersection(features), 'STOP: forbidden predictor found'
assert features == stage7['raw_features_in_order'], 'STOP: Stage 1/7 feature identity mismatch'
assert stage7['baseline_selection']['B_star'] == 'GBDT_mean', 'STOP: accepted B* mismatch'
assert stage7['dataset_sha256'] == EXPECTED_DATASET_SHA, 'STOP: Stage 7 dataset SHA mismatch'
assert stage7['working_index_sha256'] == EXPECTED_WORKING_SHA, 'STOP: Stage 7 working-index SHA mismatch'

raw = pd.read_excel(DATASET, engine='pyxlsb')
assert TARGET in raw and IDENTIFIER in raw, 'STOP: target/identifier missing'
assert all(column in raw for column in features), 'STOP: required feature missing'
with np.load(STAGE7_OOF_PATH, allow_pickle=False) as artifact:
    required = {'working_indices', 'target', 'fold', 'gbdt_mean_probability'}
    assert required.issubset(artifact.files), f'STOP: OOF keys missing: {required - set(artifact.files)}'
    working_indices = np.asarray(artifact['working_indices'], dtype=np.int64)
    y_working = np.asarray(artifact['target'], dtype=np.int8)
    fold = np.asarray(artifact['fold'], dtype=np.int8)
    gbdt_mean = np.asarray(artifact['gbdt_mean_probability'], dtype=np.float64)

assert len(working_indices) == len(y_working) == len(fold) == len(gbdt_mean) == EXPECTED_N, 'STOP: unexpected working n'
assert np.unique(working_indices).size == EXPECTED_N, 'STOP: duplicate working indices'
assert sha256_indices(working_indices) == EXPECTED_WORKING_SHA, 'STOP: working-index SHA mismatch'
assert np.array_equal(raw.loc[working_indices, TARGET].to_numpy(dtype=np.int8), y_working), 'STOP: target alignment mismatch'
assert np.isfinite(gbdt_mean).all() and ((0.0 <= gbdt_mean) & (gbdt_mean <= 1.0)).all(), 'STOP: non-finite/invalid GBDT_mean'
assert set(np.unique(fold)) == {1, 2, 3}, 'STOP: invalid fold labels'

splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=OUTER_SEED)
expected_fold = np.zeros(EXPECTED_N, dtype=np.int8)
for fold_id, (_, valid_pos) in enumerate(splitter.split(np.zeros(EXPECTED_N), y_working), start=1):
    expected_fold[valid_pos] = fold_id
assert np.array_equal(fold, expected_fold), 'STOP: saved fold assignment differs from locked outer CV'

X_working = raw.loc[working_indices, features].copy()
assert not X_working.isna().any().any(), 'STOP: RealMLP does not accept missing numericals; no imputation is permitted here'
assert all(pd.api.types.is_numeric_dtype(X_working[column]) for column in features), 'STOP: unexpected non-numeric feature'
assert np.isfinite(X_working.to_numpy(dtype=np.float64)).all(), 'STOP: non-finite feature value'
print(f'PREFLIGHT PASS: n={EXPECTED_N}, features={len(features)}, baseline finite, dataset/indices/target/folds aligned.')


### Что проверяем?

Выполняем ровно три outer folds. Для каждого RealMLP получает только X_train/y_train; его official native preprocessing и internal validation остаются fold-local. В notebook выводятся heartbeat по fold, стадии и elapsed time, а resolved constructor parameters сохраняются в result JSON. Это отвечает на вопрос сравнением свежего RealMLP OOF с уже сохранённым GBDT_mean без повторного GBDT run.

Полный запуск дорогой и запускается пользователем только после pre-run review.


In [ ]:
def decide(delta_gini: float, fold_deltas: list[float]) -> str:
    wins = sum(delta > 0.0 for delta in fold_deltas)
    losses = sum(delta < 0.0 for delta in fold_deltas)
    if delta_gini >= 0.010 and wins >= 2:
        return 'material_gain'
    if delta_gini <= -0.010 and losses >= 2:
        return 'inferior'
    return 'no_material_benefit'

def run_stage11() -> dict:
    started = time.monotonic()
    realmlp_oof = np.full(EXPECTED_N, np.nan, dtype=np.float64)
    fold_rows, resolved_configs = [], []
    for fold_id, seed in enumerate(FOLD_SEEDS, start=1):
        fold_started = time.monotonic()
        train_pos, valid_pos = np.flatnonzero(fold != fold_id), np.flatnonzero(fold == fold_id)
        print(f'[Этап 11] Fold {fold_id}/3 | стадия: RealMLP native fit | elapsed={time.monotonic() - started:.1f}с')
        model = RealMLP_TD_Classifier(random_state=seed, **MODEL_OVERRIDES)
        resolved = json_safe(model.get_params(deep=False))
        resolved_configs.append({'fold': fold_id, 'seed': seed, 'params': resolved})
        model.fit(X_working.iloc[train_pos], y_working[train_pos])
        print(f'[Этап 11] Fold {fold_id}/3 | стадия: predict outer-validation | elapsed={time.monotonic() - started:.1f}с')
        probability = np.asarray(model.predict_proba(X_working.iloc[valid_pos])[:, 1], dtype=np.float64)
        assert np.isfinite(probability).all() and ((0.0 <= probability) & (probability <= 1.0)).all(), 'STOP: invalid RealMLP probability'
        realmlp_oof[valid_pos] = probability
        model_metrics = metrics_at_threshold(y_working[valid_pos], probability)
        baseline_metrics = metrics_at_threshold(y_working[valid_pos], gbdt_mean[valid_pos])
        delta = {key: model_metrics[key] - baseline_metrics[key] for key in model_metrics}
        fold_rows.append({'fold': fold_id, 'seed': seed, 'n_train': int(train_pos.size), 'n_validation': int(valid_pos.size), 'runtime_seconds': time.monotonic() - fold_started, 'realmlp_metrics': model_metrics, 'gbdt_mean_metrics': baseline_metrics, 'delta_realmlp_minus_gbdt_mean': delta})
        print(f'[Этап 11] Fold {fold_id}/3 завершён | ΔGini={delta["Gini"]:+.6f} | elapsed={time.monotonic() - started:.1f}с')

    assert np.isfinite(realmlp_oof).all(), 'STOP: incomplete RealMLP OOF'
    realmlp_metrics = metrics_at_threshold(y_working, realmlp_oof)
    baseline_metrics = metrics_at_threshold(y_working, gbdt_mean)
    deltas = {key: realmlp_metrics[key] - baseline_metrics[key] for key in realmlp_metrics}
    fold_gini_deltas = [row['delta_realmlp_minus_gbdt_mean']['Gini'] for row in fold_rows]
    result = {
        'experiment': 'Stage 11', 'version': 'V1', 'status': 'completed',
        'dataset_sha256': EXPECTED_DATASET_SHA, 'working_index_sha256': EXPECTED_WORKING_SHA,
        'target': TARGET, 'identifier': IDENTIFIER, 'raw_features_in_order': features,
        'raw_feature_count': len(features), 'forbidden_features': list(FORBIDDEN),
        'pytabkit_version': importlib.metadata.version('pytabkit'),
        'model': 'RealMLP_TD_Classifier', 'model_overrides': MODEL_OVERRIDES,
        'resolved_realmlp_config_by_fold': resolved_configs,
        'outer_cv': {'type': 'StratifiedKFold', 'n_splits': 3, 'shuffle': True, 'random_state': OUTER_SEED},
        'outer_fold_seeds': {str(index): seed for index, seed in enumerate(FOLD_SEEDS, start=1)},
        'baseline': {'name': 'GBDT_mean', 'source': str(STAGE7_OOF_PATH.relative_to(ROOT)), 'retrained': False},
        'protocol': {'device': 'cpu', 'n_cv': 1, 'n_refit': 0, 'n_ens': 1, 'hpo': False, 'bagging_or_ensembling': False, 'calibration': False, 'class_weighting': False, 'balancing_or_sampling': False, 'threshold_optimization': False, 'diagnostic_threshold': THRESHOLD, 'preprocessing': 'official RealMLP native fold-local pipeline'},
        'fold_metrics': fold_rows, 'realmlp_oof_metrics': realmlp_metrics, 'gbdt_mean_oof_metrics': baseline_metrics,
        'delta_realmlp_minus_gbdt_mean': deltas, 'decision': decide(deltas['Gini'], fold_gini_deltas),
        'final_test_used': False, 'runtime_seconds': time.monotonic() - started,
        'limitations': ['Random CV does not prove temporal stability.', 'Three folds are not a statistical-significance claim.', 'Precision/Recall/F1 at 0.5 are diagnostic only.', 'Final test was not used.', 'One tuned-default RealMLP recipe was evaluated without KOMUS-specific HPO.'],
    }
    atomic_npz(OOF_PATH, working_indices=working_indices, target=y_working, fold=fold, gbdt_mean_probability=gbdt_mean, realmlp_oof_probability=realmlp_oof)
    atomic_json(RESULT_PATH, result)
    atomic_json(SUMMARY_PATH, {'experiment': 'Stage 11 V1', 'decision': result['decision'], 'primary_metric': 'OOF Gini', 'realmlp_oof_gini': realmlp_metrics['Gini'], 'gbdt_mean_oof_gini': baseline_metrics['Gini'], 'delta_gini': deltas['Gini'], 'fold_gini_deltas': fold_gini_deltas, 'final_test_used': False, 'result_artifacts': [str(RESULT_PATH.relative_to(ROOT)), str(OOF_PATH.relative_to(ROOT))]})
    print(json.dumps({'OOF ΔGini': deltas['Gini'], 'decision': result['decision'], 'runtime_seconds': result['runtime_seconds']}, ensure_ascii=False, indent=2))
    return result

# После pre-run review пользователь запускает только эту строку:
# stage11_result = run_stage11()


# Результат исследования

## ФАКТЫ

Заполнить по сохранённому JSON после будущего полного run: OOF metrics RealMLP, GBDT_mean, Δ и fold evidence.

## ИНТЕРПРЕТАЦИЯ

Использовать только заранее зафиксированное decision rule по OOF Gini; Recall/F1@0.5 не меняют classification.

## ОГРАНИЧЕНИЯ

Random CV не доказывает temporal stability; три folds не являются claim о statistical significance; final test не использован.

## СЛЕДУЮЩИЙ ШАГ

Результат передаётся Technical Coordinator для решения о завершении или продолжении model-only ветки.
